# Least Squares Migration с помощью PyLops

Least Squares Migration (LSM) — метод миграции, основанный на формулировке обратной задачи как задачи наименьших квадратов. 
Цель: найти модель отражающей способности (reflectivity), которая при прохождении через прямой оператор (forward operator) воспроизводит наблюдаемые сейсмограммы.

В отличие от обычной миграции, которая применяет не обратный (inverse), а сопряженный (adjoint) оператор и дает лишь приближенное решение.

План работы:
- Построение скоростной модели и модели отражательной способности;
- Моделирование сейсмограмм с помощью прямого Кирхгофф-оператора (LSM в режиме eikonal с помощью готового оператора из из библиотеки PyLops);
- Обычную (adjoint) миграцию и итеративную LSM-миграцию методом FISTA для разреженного решения с помощью готового оператора из библиотеки PyLops;
- Сравнение результатов.

## 2. Импорт модулей

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tqdm import tqdm
import pylops
from pylops.optimization.sparsity import fista

# Настройка отображения для ноутбука
%matplotlib inline

## 3. Создание модели скоростей, отражательной способности (reflectivity) и положений источников и приемников

In [ ]:
# Параметры сетки
nx = 201
nz = 201
dx = 10
dz = 10
x = np.arange(nx) * dx
z = np.arange(nz) * dz

# Скорости слоев
v_top = 2000.0
v_middle = 2500.0
v_bottom = 3000.0

# Основная граница (глубже)
z_boundary = 1000.0
iboundary = int(z_boundary / dz)

# Параметры волнистой верхней границы
z_wave_boundary = 400.0
wave_amplitude = 60.0
wave_frequency = 0.008

# Волнистая граница и модель скоростей
wave_boundary_z = z_wave_boundary + wave_amplitude * np.sin(wave_frequency * x)
wave_boundary_idx = np.round(wave_boundary_z / dz).astype(int)

# v_model имеет форму (nz, nx): первая ось - z, вторая - x
v_model = np.ones((nz, nx)) * v_bottom
v_model[:iboundary, :] = v_top
for i in range(nx):
    wave_top = max(0, wave_boundary_idx[i])
    v_model[:wave_top, i] = v_middle

# Геометрия: источники и приемники
sx = np.arange(50, 2000, 200)
sz = np.ones_like(sx) * 10
sources = np.vstack((sx, sz)).astype(float)

rx = np.arange(0, 2025, 25)
rz = np.zeros_like(rx)
recs = np.vstack((rx, rz)).astype(float)

# Построим модель reflectivity по градиенту скоростей (в форме (nz, nx))
refl = np.zeros_like(v_model)  # shape (nz, nx): z,x
for ix in range(nx):
    vcol = v_model[:, ix]            # vertical column along z for x index ix
    rcol = np.zeros(nz)
    rcol[1:] = (vcol[1:] - vcol[:-1]) / (vcol[1:] + vcol[:-1] + 1e-9)
    refl[:, ix] = rcol

# Добавим небольшой дифрактор как в предыдущих примерах
diffractor_x = int(nx * 0.7)
diffractor_z = int(nz * 0.6)
diffractor_refl = 0.1
difw = 1
# refl indexed as [z, x]
refl[diffractor_z - difw:diffractor_z+difw+1, diffractor_x-difw:diffractor_x+difw+1] = diffractor_refl

print('Shapes: v_model', v_model.shape, 'refl', refl.shape)

## 4. Совместная визуализация модели скоростей и модели рефлективити с наложенными источниками и приемниками

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(v_model, extent=[x.min(), x.max(), z.max(), z.min()], cmap='viridis')
plt.title('Скоростная модель')
plt.scatter(sources[0], sources[1], color='red', s=60, marker='v', label='Источники')
plt.scatter(recs[0], recs[1], color='white', s=20, alpha=0.7, marker='x', label='Приемники')
plt.xlim(0, 2000)
plt.xlabel('x (м)')
plt.ylabel('z (м)')
plt.legend()
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(refl, extent=[x.min(), x.max(), z.max(), z.min()], cmap='gray')
plt.title('Reflectivity (р)')
plt.scatter(sources[0], sources[1], color='red', s=60, marker='v')
plt.scatter(recs[0], recs[1], color='blue', s=20, alpha=0.6)
plt.xlim(0, 2000)
plt.xlabel('x (м)')
plt.ylabel('z (м)')
plt.colorbar()
plt.tight_layout()
plt.show()

## 5. Сглаживание модели. Визуализация

In [ ]:
# Сглаживание фоновой скорости (background) для использования в LSM-операторе
# v_smooth имеет форму (nz, nx) — первая ось z, вторая x
v_smooth = gaussian_filter(v_model, sigma=20)

plt.figure(figsize=(6, 5))
plt.imshow(v_smooth, extent=[x.min(), x.max(), z.max(), z.min()], cmap='viridis')
plt.title('Сглаженная (фоновая) скоростная модель')
plt.xlabel('x (м)')
plt.ylabel('z (м)')
plt.colorbar()
plt.show()

## 6. Моделирование сейсмограмм с помощью прямого Kirchoff-оператора. Визуализация.

In [ ]:
# Временные параметры и вейвлет
nt = 651
dt = 0.004
t = np.arange(nt) * dt
wav, wavt, wavc = pylops.utils.wavelets.ricker(t[:41], f0=20)

print('Building LSM operator (mode=eikonal)...')
# Передаём фон в форме (nx, nz) — поэтому транспонируем v_smooth при подаче в LSM
lsm = pylops.waveeqprocessing.LSM(
    z,
    x,
    t,
    sources,
    recs,
    v_smooth.T,     # background velocity field (transposed to shape (nx, nz) as expected by LSM)
    wav,
    wavc,
    mode='eikonal',
    engine='numba',
)
print('LSM operator created')

# Прямое моделирование (векторно)
ns = sources.shape[1]
nr = recs.shape[1]

print('Modeling all data at once (vectorized call)...')
# Важно: LSM ожидает вектор модели в порядке (nx, nz). Мы храним refl в порядке (nz, nx),
# поэтому перед вызовом оператора транспонируем (refl.T.ravel())
d_all = (lsm.Demop * refl.ravel()).reshape(ns, nr, nt)

# Для наглядности разобьем в цикле (можно пропустить для ускорения)
d = np.zeros_like(d_all)
for i in tqdm(range(ns), desc='Sources'):
    d[i, :, :] = d_all[i, :, :]

# Показать один источник — пример съемки (gather)
src_idx = len(sources[0]) // 2
plt.figure(figsize=(8, 8))
plt.imshow(d[src_idx].T, aspect='auto', cmap='gray', extent=[0, nr, t.max(), t.min()])
plt.title(f'Синтетическая съемка для источника {src_idx}')
plt.xlabel('Приемник #')
plt.ylabel('Время (с)')
plt.colorbar(label='Амплитуда')
plt.show()

## 7. Обычная adjoint-миграция. Визуализция результата с сравнении с моделью скорости и рефлективити

In [ ]:
# Соберем вектор данных и выполним adjoint-миграцию
dvec = d.ravel()
print('Performing adjoint migration...')
madj = lsm.Demop.H * dvec
madj = madj.reshape(nx, nz)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(madj, cmap='gray', extent=[x.min(), x.max(), z.max(), z.min()])
plt.title('Adjoint migration')
plt.xlabel('x (м)')
plt.ylabel('z (м)')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(refl, cmap='gray', extent=[x.min(), x.max(), z.max(), z.min()])
plt.title('Reflectivity (истинная)')
plt.xlabel('x (м)')
plt.ylabel('z (м)')
plt.colorbar()
plt.tight_layout()
plt.show()

## 8. LSM-миграция методом FISTA

🎯 FISTA (Fast Iterative Shrinkage-Thresholding Algorithm) — это не просто LSM (Least Squares Migration), а метод для решения задачи Least Squares с $L_1$-регуляризацией, также известной как Lasso или Sparse Least Squares. 

Его основное назначение — обеспечить разреженность (sparsity) в результирующей модели.

1. 🔍 Что делает FISTA (Целевая функция)
Обычная минимизация по методу наименьших квадратов (Least Squares, LS), которую решает, например, LSQR, минимизирует только ошибку данных:

$$\text{LS: } \min_{\mathbf{m}} \|\mathbf{L}\mathbf{m} - \mathbf{d}\|_2^2$$

FISTA же решает оптимизационную задачу с добавлением $L_1$-нормы модели, взвешенной коэффициентом $\lambda$ (который вы задали как eps):

$$\text{FISTA (Lasso): } \min_{\mathbf{m}} \underbrace{\|\mathbf{L}\mathbf{m} - \mathbf{d}\|_2^2}_{\text{Ошибка данных } (L_2)} + \underbrace{\lambda \|\mathbf{m}\|_1}_{\text{Штраф за разреженность } (L_1)}$$

2. ✨ Зачем нужна $L_1$-регуляризация (Разреженность)
В геофизике и сейсмике мы часто предполагаем, что отражающая способность Земли $\mathbf{m}$ является разреженной — то есть, отражатели существуют только на нескольких четких границах (контрастах скоростей), а не размазаны по всему объему.

Добавление штрафа $L_1$:

- Обеспечивает разреженность: Штраф $\|\mathbf{m}\|_1$ (сумма абсолютных значений элементов модели) активно заставляет малые элементы $\mathbf{m}$ становиться нулем (это достигается с помощью операции мягкого порога — soft thresholding).
- Улучшает фокусировку: Результат LSM с $L_1$-регуляризацией (Sparse LSM) часто дает более четкие и сфокусированные отражающие границы по сравнению с обычным LSM.
- Снижает шум: $L_1$ помогает подавлять артефакты и шум, которые обычно присутствуют в решении LSM.

In [ ]:
# Настройки FISTA. В Pylops уже реализован оператор для FISTA 
eps = 1e2
niter = 70
eigsdict = dict(niter=5, tol=1e-2)
show = True

print(f'Running FISTA for {niter} iterations (eps={eps})...')
minv_vec, cost, resnorm = fista(lsm.Demop, dvec, eps=eps, niter=niter, eigsdict=eigsdict, show=show)
minv = minv_vec.reshape(nz, nx)
print('FISTA finished')

## 9. Финальная визуализация: модель скоростей, модель рефлективити, результат Adjoint, результат LSM

In [ ]:
plt.figure(figsize=(12, 10))

plt.subplot(2, 2, 1)
plt.imshow(v_model, cmap='viridis', extent=[x.min(), x.max(), z.max(), z.min()])
plt.title('Скоростная модель')
plt.colorbar()

plt.subplot(2, 2, 2)
plt.imshow(refl, cmap='gray', extent=[x.min(), x.max(), z.max(), z.min()])
plt.title('Reflectivity (истинная)')
plt.colorbar()

plt.subplot(2, 2, 3)
plt.imshow(madj, cmap='gray', extent=[x.min(), x.max(), z.max(), z.min()])
plt.title('Adjoint migration')
plt.colorbar()

plt.subplot(2, 2, 4)
plt.imshow(minv, cmap='gray', extent=[x.min(), x.max(), z.max(), z.min()])
plt.title(f'LSM inversion (FISTA) {niter} итераций')
plt.colorbar()

plt.tight_layout()
plt.show()

## Анализ результатов

Очевидно, на изображении LSM-миграции видна гораздо более чёткая, сфокусированная картинка. 
Можно попробовать уменьшить количество итераций и найти баланс время выполнения/качество